In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

# From different folder Importing defined schemas

In [0]:
%run
/Workspace/Users/kharitejareddy997@gmail.com/setup/utlities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
%fs ls "s3://orders-ecommerce-de/raw"

path,name,size,modificationTime
s3://orders-ecommerce-de/raw/customers.csv,customers.csv,245761,1782049384000
s3://orders-ecommerce-de/raw/products.csv,products.csv,83070,1782049384000
s3://orders-ecommerce-de/raw/sales_reps.csv,sales_reps.csv,20062,1782049384000
s3://orders-ecommerce-de/raw/transactions.csv,transactions.csv,625057,1782049384000


In [0]:
# Specifing the location path
path = 's3://orders-ecommerce-de/raw/customers.csv'
df = spark.read.csv(path, header=True, inferSchema=True)

# Writing this data from S3 Bucket to Bronze Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('ecommerce.bronze.customers')

In [0]:
df_silver = spark.read.table('ecommerce.bronze.customers')

# Applying Data Transformations

In [0]:
# Checking For Duplicates
display(df_silver.groupBy("customer_id").count().filter(f.col("count")>1))

customer_id,count
CUST_0006,2
CUST_0013,2
CUST_0033,2
CUST_0055,2
CUST_0092,2
CUST_0171,2
CUST_0172,2
CUST_0178,2
CUST_0192,2
CUST_0196,2


In [0]:
# Droping Duplicates
df_silver = df_silver.dropDuplicates(['customer_id'])

In [0]:
# Trimming the white spaces in customer_name column 
df_silver = df_silver.withColumn('customer_name', f.trim(f.col('customer_name')))

In [0]:
# Converting all the emails into lower case values
df_silver = df_silver.withColumn('email', f.lower(f.col('email')))


In [0]:
# Removing all the digits in the phone column
df_silver = df_silver.withColumn('phone', f.regexp_replace(f.col('phone'), r'\D', ''))

In [0]:
# Fixing The Inconsistent date formats
df_silver = df_silver.withColumn('registration_date', 
                             f.coalesce(
                                 f.try_to_date(f.col("registration_date"), 'dd/MM/yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'MM/dd/yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy/MM/dd'),
                                 f.try_to_date(f.col("registration_date"), 'dd-MM-yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'MM-dd-yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy-dd-MM'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy.dd.MM'),
                                 f.try_to_date(f.col("registration_date"), 'dd.MM.yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'MM.dd.yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy.MM.dd'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy-dd-MMMM'),
                                 f.try_to_date(f.col("registration_date"), 'MMMM-dd-yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'dd-MMMM-yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy-dd-MMM'),
                                 f.try_to_date(f.col("registration_date"), 'MMM-dd-yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'dd-MMM-yyyy'),
                                 f.try_to_date(f.col("registration_date"), 'yyyy-MM-dd')
                                 ))


In [0]:
# Fixing Inconsistent country names
df_silver = df_silver.withColumn('country', 
                     f.when((f.col("city") == 'Lyon') | (f.col("city") == 'Paris') | (f.col("city")=='France'), 'France')
                     .when((f.col("city") == 'Manchester') | (f.col("city") == 'London'), 'UK')
                     .when(((f.col('city') == 'New York')) | (f.col('city') == 'Los Angeles') | (f.col('city') == 'Chicago'), 'USA')
                     .when((f.col("city") == 'Toronto'), 'Canada')
                     .when(f.col("city") == 'Berlin','Germanay')
                     
                     )

In [0]:
# Fixing Inconsistent city names
df_silver = df_silver.replace('France', 'Paris', subset=["city"])

In [0]:
# Fixing Inconsistent pincodes
df_silver = df_silver.withColumn('postal_code', 
                     f.when(f.col("city") == 'Chicago', '60601')
                     .when(f.col('city') == 'Berlin', '10115')
                     .when(f.col("city") == 'France', '75001')
                     .when(f.col("city") == 'Toronto', 'M5H 2N2')
                     .when(f.col("city") == 'Lyon', '69001')
                     .when(f.col("city") == 'Los Angeles', '90001')
                     .when(f.col("city") == 'New York', '10001')
                     .when(f.col("city") == 'Paris', '10115')
                     .when(f.col("city") == 'Manchester', 'M1 1AE')
                     .when(f.col("city") == 'London', 'SWA1A 1AA')
                    
                     )

In [0]:
# Replacing Null Email's with name + @gmail.com
df_silver = df_silver.withColumn('email', 
                     f.when(f.col('email').isNull(), f.lower(f.concat(f.regexp_replace(f.col('customer_name'),' ', '.'), f.lit('@gmail.com'))))
                     .when(f.col('email') == 'invalid.email', f.lower(f.concat(f.regexp_replace(f.col('customer_name'),' ', '.'), f.lit('@gmail.com'))))
                     .otherwise(f.col('email'))
                     )

In [0]:
# Replacing all null values in phone column with default number '99999999999'
df_silver = df_silver.fillna('9999999999', subset=['phone'])

In [0]:
# Rounding the lifetime_value column to 2 decimal places
df_silver = df_silver.withColumn('lifetime_value', f.round(f.col("lifetime_value"), 2))

In [0]:
# Converting phone datatype
df_silver = df_silver.withColumn('phone', f.col("phone").cast('bigint'))


In [0]:
display(df_silver)

customer_id,customer_name,email,phone,registration_date,customer_segment,country,city,postal_code,lifetime_value
CUST_0001,Jessica Brown,jessica.brown41@gmail.com,15558337628,2022-07-11,VIP,UK,London,SWA1A 1AA,9701.21
CUST_0002,William Johnson,william.johnson16@gmail.com,9999999999,2022-01-29,Retail,UK,Manchester,M1 1AE,7303.55
CUST_0003,Robert Rodriguez,robert.rodriguez6@gmail.com,5552034602,2023-08-18,Wholesale,USA,Chicago,60601,9233.78
CUST_0004,David Miller,david.miller17@gmail.com,5558515395,2022-06-14,Retail,USA,New York,10001,7645.9
CUST_0005,John Williams,john.williams74@gmail.com,9999999999,2022-03-19,VIP,France,Lyon,69001,5958.0
CUST_0006,Michael Brown,michael.brown27@gmail.com,5552754335,2021-04-07,Retail,France,Lyon,69001,2001.03
CUST_0007,Sarah Miller,sarah.miller84@gmail.com,9999999999,2023-08-24,Retail,UK,Manchester,M1 1AE,6700.19
CUST_0008,Ashley Davis,ashley.davis21@gmail.com,15558225718,2022-03-30,Corporate,USA,Chicago,60601,6271.46
CUST_0009,Michael Garcia,michael.garcia12@gmail.com,5558511050,2021-01-25,Corporate,USA,Los Angeles,90001,6064.44
CUST_0010,Michael Johnson,michael.johnson72@gmail.com,5551964032,2022-12-12,Retail,France,Paris,10115,4948.88


# Writing to the Silver Layer

In [0]:
df_silver.write.mode('overwrite').option('mergeSchema', True).format('delta').saveAsTable('ecommerce.silver.customers')

# Writing to the Gold Layer

In [0]:
dim_customers = spark.sql("""
          select distinct customer_id, customer_name, email, phone, customer_segment, country, city, postal_code, registration_date 
          from ecommerce.silver.customers
          """)

In [0]:
dim_customers.write.mode('overwrite').format('delta').saveAsTable('ecommerce.gold.dim_customers')